# GeoNUSAF Baseline — SegFormer-B0 (MiT-B0, ADE20K-pretrained)

**Kathmandu Valley satellite land-use segmentation — 6 classes + ignore**

Structural twin of `unet.ipynb`: same 25 sections, same config-dataclass pattern, same asserts,
same persistence. Only the model section and the regularization block differ, so the two notebooks
diff cleanly.

## What is fixed by protocol (do not change per-model)
| Item | Value |
|---|---|
| Split | 3-fold CV over **408 base tiles** (flip variants travel as one group) |
| Split modes | `block` (export-sequence proxy) **and** `random` (leakage-gap control) |
| Loss | effective-number-weighted CE + Lovasz-Softmax, lambda=0.75 |
| Optimizer | AdamW, differential LR (encoder = head/10) |
| Schedule | 500-step linear warmup then cosine decay |
| Stability | `clip_grad_norm_(1.0)`, grad-norm logged every step |
| Normalization | ImageNet mean/std |
| Augmentation | rot90 (k in 0..3) + mild photometric jitter, **on-the-fly**. hflip/vflip are already baked into the dataset and are NOT repeated. |
| Epochs | max 200, early stop on val mIoU, patience 25 |
| Seed | single seed = 42 |
| Ignore | index 255, excluded from loss and all metrics |
| Batch | 16, `GRAD_ACCUM=1` |

## What is model-specific here
- `SegformerForSemanticSegmentation`, `nvidia/segformer-b0-finetuned-ade-512-512` (~3.8 M params)
- LR: head 6e-5 / encoder 6e-6. The earlier exploratory run used a flat 3e-5 and probably
  undertrained; 6e-5 is the SegFormer paper's ADE20K fine-tuning default. Watch val mIoU over the
  first ~10 epochs before locking it in.
- **Logits come out at H/4 = 128x128** and are bilinearly upsampled to 512 before the loss. This is
  the SegFormer convention and it is the single easiest thing to get wrong when porting the U-Net
  training loop — every forward pass goes through `forward_logits()` for exactly that reason.
- LayerNorm/GroupNorm throughout, so no BatchNorm small-batch sensitivity.

## Response to the U-Net overfitting curves
Your U-Net `val_loss` bottoms around epoch 8-10 and then climbs to ~2.8 while `train_loss` falls to
~0.05. Two things are true at once, and it matters which one you act on:

1. **Real memorization.** A train loss of 0.05 on ~816 images from 272 independent scenes is a
   near-perfect fit. That part is genuine and worth regularizing.
2. **Rising val loss overstates it.** You early-stop on val **mIoU**, and cross-entropy keeps
   growing whenever the model becomes *more confident* on the pixels it already gets wrong — even
   while argmax accuracy still improves. Fold f2 ran to ~epoch 72 with patience 25, so its best
   val mIoU was around epoch 47, roughly 40 epochs *after* the val-loss minimum. **Before changing
   anything, plot `val_mIoU` from `history_{mode}_fold{k}.csv` and find where it actually peaked.**
   If it peaked near epoch 10, the overfitting is severe; if it peaked at 45-85, the loss curve is
   mostly a calibration artifact and the fix below is a refinement, not a rescue.

### Changes in this notebook, and which ones oblige you to re-run U-Net
| Knob | U-Net run | Here | Category |
|---|---|---|---|
| LR (head / encoder) | 1e-4 / 1e-5 | 6e-5 / 6e-6 | per-architecture — no re-run |
| weight decay | 1e-4 | **0.01** | per-architecture — no re-run |
| no decay on norm/bias | n/a | **on** | per-architecture — no re-run |
| stochastic depth (`drop_path_rate`) | n/a (CNN) | **0.1** | per-architecture — no re-run |
| head dropout | n/a | **0.1** | per-architecture — no re-run |
| CE label smoothing | 0.0 | **0.05** | **shared — back-apply to every finished baseline** |
| cosine horizon | 200 epochs | **120 epochs** | **shared — back-apply** |
| weight EMA | none | **0.999** | **shared — back-apply** |
| max epochs / patience / batch / aug | 200 / 25 / 16 / rot90+jitter | unchanged | unchanged |

The split into "per-architecture" and "shared" is the defensible line for a benchmark table: LR,
weight decay and an architecture's own published dropout rates follow each paper's fine-tuning
recipe (reviewers expect this — a fixed LR across a CNN and a transformer is the *less* defensible
choice), while the loss function, schedule shape, and weight-averaging scheme are part of the
protocol and must be identical everywhere. `LABEL_SMOOTHING=0.0`, `COS_HORIZON_EPOCHS=200`,
`USE_EMA=False` reproduces the U-Net protocol exactly if you would rather not re-run anything —
decide once, then never mix.

## Session strategy (12 h Kaggle cap)
Each **fold is one session**. Full optimizer/scaler/scheduler/EMA state is pushed to Hugging Face
as `last.pt`; on restart the notebook resumes from it automatically. Set `FOLD` and `SPLIT_MODE`
in the config cell, run top to bottom, repeat.

## Prerequisites
- **Internet ON** (Settings -> Internet). Needed for pip, ADE20K weights, W&B, HF.
- Kaggle Secrets: `HF_TOKEN` (write scope) and `WANDB_API_KEY`.
- GPU: T4 x1.

## 1. Environment

`transformers` is preinstalled on Kaggle; only `ptflops` is not. The transformers version is
printed rather than pinned — `SegformerForSemanticSegmentation` has had a stable API since 4.12,
but record the printed version in the paper's reproducibility section.

In [ ]:
!pip install -q ptflops==0.7.3 2>&1 | tail -2

import os, re, gc, json, time, math, random, warnings, itertools, copy
from dataclasses import dataclass, asdict, field
from pathlib import Path
from typing import List, Dict, Tuple, Optional

import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")
cv2.setNumThreads(0)   # critical: cv2 threads fight DataLoader workers on Kaggle's 4 vCPUs

import transformers
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| cuda", torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

## 2. Configuration

**Set `FOLD` and `SPLIT_MODE` here, then run the whole notebook.** One fold per session.

`SPLIT_MODE`:
- `"block"` — contiguous runs of export-sequence order as a spatial-block proxy (the reported protocol)
- `"random"` — fully random base-tile split (the leakage-gap control)

Both still group flip variants, so neither has *duplicate* leakage; the difference between them
isolates *spatial* leakage specifically.

The regularization block is grouped and commented separately so you can see at a glance what
differs from the U-Net run, and so a later ablation ("B0 without EMA/label smoothing") is a
one-line change rather than a code edit.

In [ ]:
@dataclass
class CFG:
    # ---- run selection (change these per session) ----
    SPLIT_MODE: str = "block"          # "block" | "random"
    FOLD: int = 0                      # 0 | 1 | 2

    # ---- data ----
    DATA_ROOT: str = "/kaggle/input/datasets/sugam24/satellite-datasets/satellite_dataset"
    CACHE_DIR: str = "/kaggle/working/cache512"
    # If you saved the repacked cache as a Kaggle dataset, point this at it to skip the repack.
    PREBUILT_CACHE: Optional[str] = None
    IMG_SIZE: int = 512
    N_CLASSES: int = 6
    IGNORE_INDEX: int = 255
    PRELOAD_RAM: bool = True           # ~1.3 GB for the whole 512 cache; kills all disk I/O

    # ---- split ----
    N_FOLDS: int = 3
    BLOCK_BUFFER: int = 2              # base tiles dropped from train at each block boundary

    # ---- model ----
    HF_MODEL_ID: str = "nvidia/segformer-b0-finetuned-ade-512-512"
    DROP_PATH_RATE: float = 0.1        # stochastic depth (MiT default; raise to 0.2 if still overfitting)
    CLASSIFIER_DROPOUT: float = 0.1    # dropout on the all-MLP decode head
    HIDDEN_DROPOUT: float = 0.0        # SegFormer default; left off (drop_path is the main lever)
    ATTENTION_DROPOUT: float = 0.0

    # ---- optimization ----
    BATCH_SIZE: int = 16
    GRAD_ACCUM: int = 1                # raise to 2 and halve BATCH_SIZE if you OOM (B0 will not)
    LR_DECODER: float = 6e-5           # decode head; SegFormer's ADE20K fine-tuning default
    LR_ENCODER: float = 6e-6           # 10x differential, per protocol
    WEIGHT_DECAY: float = 0.01         # SegFormer paper value (U-Net used 1e-4)
    NO_DECAY_ON_NORM_BIAS: bool = True # never decay LayerNorm gains or biases
    MAX_EPOCHS: int = 200
    PATIENCE: int = 25
    WARMUP_STEPS: int = 500
    CLIP_NORM: float = 1.0
    LAMBDA_LOVASZ: float = 0.75
    EN_BETA: float = 0.9999            # effective-number beta (Cui et al. 2019)
    WEIGHT_CAP: float = 10.0           # cap on normalized class weights

    # ---- regularization added after the U-Net overfitting curves ----
    # Set LABEL_SMOOTHING=0.0, COS_HORIZON_EPOCHS=MAX_EPOCHS, USE_EMA=False to reproduce the
    # exact U-Net protocol. Whatever you choose must be identical across all seven baselines.
    LABEL_SMOOTHING: float = 0.05      # CE term only; caps the confidence blow-up on wrong pixels
    COS_HORIZON_EPOCHS: int = 120      # cosine reaches ~0 here instead of at epoch 200
    USE_EMA: bool = True               # validate + checkpoint an EMA of the weights
    EMA_DECAY: float = 0.999

    # ---- augmentation (identical across all 7 baselines) ----
    AUG_ROT90: bool = True
    AUG_PHOTOMETRIC: bool = True
    AUG_BRIGHTNESS: float = 0.10
    AUG_CONTRAST: float = 0.10

    # ---- runtime ----
    SEED: int = 42
    NUM_WORKERS: int = 4
    AMP: bool = True

    # ---- persistence ----
    HF_USER: str = "sugam24"           # <-- set to your HF username
    HF_PREFIX: str = "geonusaf-segformer-b0"
    PUSH_LAST_EVERY: int = 10          # epochs between full-state pushes (~50 MB each for B0)
    WANDB_PROJECT: str = "geonusaf-baselines"
    OUT_DIR: str = "/kaggle/working/out"

    CLASS_NAMES: tuple = ("Residential", "Road", "River", "Forest", "UnusedLand", "Agricultural")

cfg = CFG()
os.makedirs(cfg.OUT_DIR, exist_ok=True)

RUN_NAME = f"{cfg.HF_PREFIX}-{cfg.SPLIT_MODE}-fold{cfg.FOLD}"
HF_REPO  = f"{cfg.HF_USER}/{RUN_NAME}"
print("RUN:", RUN_NAME, "| HF repo:", HF_REPO)
print(f"reg: label_smooth={cfg.LABEL_SMOOTHING} cos_horizon={cfg.COS_HORIZON_EPOCHS} "
      f"ema={cfg.USE_EMA} drop_path={cfg.DROP_PATH_RATE} wd={cfg.WEIGHT_DECAY}")

## 3. Determinism

`use_deterministic_algorithms(warn_only=True)` is deliberate: SegFormer upsamples logits with
bilinear `interpolate`, whose backward has no deterministic CUDA kernel, and `warn_only=False`
would hard-crash on the first backward pass. Everything controllable (Python/NumPy/torch RNG,
cuDNN, worker seeding, generator) is pinned, so runs are reproducible up to those kernels. State
that caveat in the paper rather than claiming bit-exact reproducibility.

In [ ]:
def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)

def worker_init_fn(worker_id):
    s = cfg.SEED + worker_id
    np.random.seed(s); random.seed(s)

set_seed(cfg.SEED)
g = torch.Generator(); g.manual_seed(cfg.SEED)
print("seeded:", cfg.SEED)

## 4. Filename parsing and base-tile grouping

Confirmed convention: `image{N}.png` / `image{N}hf.png` / `image{N}vf.png`, masks mirror with the
`mask` prefix. `N` is ascending **export order with gaps** (mono-class tiles were dropped after
export), and the three variants of a tile are *not* adjacent in a sorted listing — so grouping is
regex-based on `N`, never positional.

The assertions here are the guard rail: if a future dataset version renames anything, this cell
fails loudly instead of silently splitting a tile across folds.

In [ ]:
IMG_DIR  = Path(cfg.DATA_ROOT) / "images"
MASK_DIR = Path(cfg.DATA_ROOT) / "masks"
PAT = re.compile(r"^image(\d+)(hf|vf)?\.png$")

records = []          # (base_id:int, variant:str, image_name:str, mask_name:str)
for f in sorted(os.listdir(IMG_DIR)):
    m = PAT.match(f)
    assert m, f"unparsed image filename: {f}"
    base_id, variant = int(m.group(1)), (m.group(2) or "orig")
    mask_name = "mask" + f[len("image"):]
    assert (MASK_DIR / mask_name).exists(), f"missing mask for {f}"
    records.append((base_id, variant, f, mask_name))

groups: Dict[int, List[tuple]] = {}
for r in records:
    groups.setdefault(r[0], []).append(r)

BASE_IDS = sorted(groups)                      # ascending export order = the spatial proxy axis
assert len(records) == 1224, len(records)
assert len(BASE_IDS) == 408, len(BASE_IDS)
assert all(len(v) == 3 for v in groups.values())

gaps = [b - a for a, b in zip(BASE_IDS, BASE_IDS[1:]) if b - a > 1]
print(f"{len(records)} files | {len(BASE_IDS)} base tiles | id range {BASE_IDS[0]}..{BASE_IDS[-1]}")
print(f"{len(gaps)} gaps in export sequence (dropped tiles), total {sum(g-1 for g in gaps)} missing ids")

## 5. Mask colour diagnostic — run this before anything else

Your masks are RGB PNGs; the class table you supplied is in **BGR**. `cv2.imread` returns BGR, so
the keys apply directly — but note that `(128,0,0)` = Residential and `(0,0,128)` = River are exact
reverses of each other. **Getting the channel order wrong silently swaps your dominant class with
your rarest one** and every metric downstream would still look plausible.

The check below is the safeguard: it dumps every distinct colour actually present and then asserts
that Residential occupies far more pixels than River. It also surfaces stray colours (CVAT polygon
edge artifacts, resampling residue) — anything not in the 6-colour table becomes `ignore_index=255`.

This also addresses the device-side assert you hit on the earlier SegFormer run: nothing reaches the
loss unless it is in `{0..5, 255}`.

In [ ]:
BGR_TO_CLASS = {
    (128, 0, 0): 0,    # Residential Area
    (0, 128, 0): 1,    # Road
    (0, 0, 128): 2,    # River
    (0, 128, 128): 3,  # Forest
    (128, 128, 0): 4,  # Unused Land
    (128, 0, 128): 5,  # Agricultural Area
}

# Pack BGR into a single uint32 key so the mapping is one vectorized lookup, not 6 boolean passes.
def pack_bgr(a: np.ndarray) -> np.ndarray:
    a = a.astype(np.uint32)
    return (a[..., 0] << 16) | (a[..., 1] << 8) | a[..., 2]

LUT = np.full(1 << 24, 255, dtype=np.uint8)          # 16 MB; default = ignore
for bgr, idx in BGR_TO_CLASS.items():
    LUT[pack_bgr(np.array(bgr, dtype=np.uint8).reshape(1, 1, 3))[0, 0]] = idx

def rgbmask_to_index(bgr_img: np.ndarray) -> np.ndarray:
    return LUT[pack_bgr(bgr_img)]

# ---- diagnostic over a random sample of raw full-res masks ----
rng = np.random.RandomState(cfg.SEED)
sample = [records[i] for i in rng.choice(len(records), 40, replace=False)]
colour_hits, class_px = {}, np.zeros(7, dtype=np.int64)   # 7th bucket = ignore

for _, _, _, mname in sample:
    m = cv2.imread(str(MASK_DIR / mname), cv2.IMREAD_COLOR)   # BGR
    cols, cnts = np.unique(m.reshape(-1, 3), axis=0, return_counts=True)
    for c, n in zip(cols, cnts):
        colour_hits[tuple(int(x) for x in c)] = colour_hits.get(tuple(int(x) for x in c), 0) + int(n)
    idx = rgbmask_to_index(m)
    b = np.bincount(np.where(idx == 255, 6, idx).ravel(), minlength=7)
    class_px += b

known = set(BGR_TO_CLASS)
print("distinct BGR colours found in sample:")
for c, n in sorted(colour_hits.items(), key=lambda kv: -kv[1]):
    tag = f"-> class {BGR_TO_CLASS[c]} ({cfg.CLASS_NAMES[BGR_TO_CLASS[c]]})" if c in known else "-> IGNORE"
    print(f"  BGR{str(c):<16} {n/sum(colour_hits.values())*100:6.2f}%  {tag}")

tot = class_px.sum()
print("\nclass pixel share (sample):")
for i, nm in enumerate(list(cfg.CLASS_NAMES) + ["<ignore>"]):
    print(f"  {i if i < 6 else 255:>3} {nm:<14} {class_px[i]/tot*100:6.2f}%")

# Orientation guard: Residential must dominate River. If this trips, your channel order is flipped.
assert class_px[0] > class_px[2] * 3, (
    "Residential <= River -- BGR/RGB channel order is probably inverted. STOP and fix before training.")
print("\nchannel-order check PASSED")

## 6. One-time repack to a 512x512 cache

Your tiles are **2345x2351 on disk (~10 MB each)**. Decoding 5.5 MP PNGs and resizing them on
Kaggle's 4 vCPUs would leave the T4 idle for most of every epoch — and B0 is the *fastest* model in
your lineup, so it is the one where an I/O-bound loader hurts most: the GPU step is short enough
that data loading would dominate wall-clock entirely.

The repack does the resize **once**:
- **Images**: `INTER_AREA` (the correct kernel for 4.6x downsampling; bilinear aliases thin roads).
- **Masks**: colour -> class index at **full resolution first**, then `INTER_NEAREST` to 512.
  Resizing the RGB mask first would interpolate between class colours and manufacture phantom
  classes at every polygon edge.

Result: ~700 MB instead of 13 GB. Note the effective GSD after this step is 300 m / 512 px =
**0.586 m/px**, which is the figure to report; the native export is ~0.128 m/px.

**Run once, then `Save Version` -> add the output as a dataset -> set `PREBUILT_CACHE` to its path**
so later sessions skip this entirely (~15-20 min saved per session). If you already built this cache
for the U-Net runs, point `PREBUILT_CACHE` at it now — it is byte-identical.

In [ ]:
from concurrent.futures import ProcessPoolExecutor

CACHE = Path(cfg.PREBUILT_CACHE) if cfg.PREBUILT_CACHE else Path(cfg.CACHE_DIR)
MARKER = CACHE / "_complete.json"

def _repack_one(rec):
    _, _, iname, mname = rec
    img = cv2.imread(str(IMG_DIR / iname), cv2.IMREAD_COLOR)
    img = cv2.resize(img, (cfg.IMG_SIZE, cfg.IMG_SIZE), interpolation=cv2.INTER_AREA)
    msk = cv2.imread(str(MASK_DIR / mname), cv2.IMREAD_COLOR)
    lab = LUT[pack_bgr(msk)]                                    # full-res -> class index
    lab = cv2.resize(lab, (cfg.IMG_SIZE, cfg.IMG_SIZE), interpolation=cv2.INTER_NEAREST)
    cv2.imwrite(str(CACHE / "images" / iname), img, [cv2.IMWRITE_PNG_COMPRESSION, 3])
    cv2.imwrite(str(CACHE / "labels" / mname), lab, [cv2.IMWRITE_PNG_COMPRESSION, 3])
    return np.bincount(np.where(lab == 255, 6, lab).ravel(), minlength=7)

if MARKER.exists():
    meta = json.load(open(MARKER))
    print(f"cache present at {CACHE} ({meta['n']} pairs) -- skipping repack")
else:
    assert not cfg.PREBUILT_CACHE, f"PREBUILT_CACHE set but {MARKER} missing"
    (CACHE / "images").mkdir(parents=True, exist_ok=True)
    (CACHE / "labels").mkdir(parents=True, exist_ok=True)
    t0 = time.time(); counts = {}
    with ProcessPoolExecutor(max_workers=4) as ex:
        for rec, b in zip(records, ex.map(_repack_one, records, chunksize=8)):
            counts[rec[2]] = b.tolist()
            if len(counts) % 200 == 0:
                el = time.time() - t0
                print(f"  {len(counts)}/{len(records)}  {el:.0f}s  eta {el/len(counts)*(len(records)-len(counts)):.0f}s")
    json.dump({"n": len(counts), "counts": counts}, open(MARKER, "w"))
    print(f"repacked {len(counts)} pairs in {time.time()-t0:.0f}s")

PIX_COUNTS = {k: np.array(v, dtype=np.int64) for k, v in json.load(open(MARKER))["counts"].items()}
sz = sum(f.stat().st_size for f in CACHE.rglob('*.png')) / 1e9
print(f"cache size: {sz:.2f} GB")

## 7. Load the cache into RAM

1224 x (512x512x3 + 512x512) uint8 = **~1.3 GB**, comfortably inside Kaggle's ~13 GB. Holding it
resident removes disk I/O from the training loop entirely and lets `num_workers` do nothing but
augmentation. Set `PRELOAD_RAM=False` if you ever hit memory pressure.

In [ ]:
RAM = {}
if cfg.PRELOAD_RAM:
    t0 = time.time()
    for i, (_, _, iname, mname) in enumerate(records):
        img = cv2.imread(str(CACHE / "images" / iname), cv2.IMREAD_COLOR)
        lab = cv2.imread(str(CACHE / "labels" / mname), cv2.IMREAD_GRAYSCALE)
        RAM[iname] = (img, lab)
        if (i + 1) % 400 == 0: print(f"  loaded {i+1}/{len(records)}")
    mb = sum(a.nbytes + b.nbytes for a, b in RAM.values()) / 1e6
    print(f"preloaded {len(RAM)} pairs, {mb:.0f} MB, {time.time()-t0:.0f}s")
    # sanity: labels must only ever contain 0..5 or 255
    u = np.unique(np.concatenate([b[::32, ::32].ravel() for _, b in RAM.values()]))
    assert set(u.tolist()) <= set(range(6)) | {255}, f"unexpected label values: {u}"
    print("label value check PASSED:", u.tolist())

## 8. Splitters — sequence-block proxy and random control

### The proxy, and its honest limitation
You have no surviving grid indices, so contiguous runs of export order stand in for spatial blocks.
QGIS Atlas iterates the coverage layer in feature order, which for a rectangular grid is normally
**row-major**. That means:

- Consecutive ids are neighbours **along the scan axis only**. Adjacency *perpendicular* to it
  (the tile directly above/below) is completely uncontrolled — those tiles can land in different folds.
- Every row wrap is a spatial discontinuity, so a "block" is really a set of strips.
- Dropped mono-class tiles break runs, so id distance overstates ground distance near gaps.

Net effect: **this proxy under-estimates true spatial leakage** relative to real geo-blocked CV.
Report it as *sequence-block CV (export-order proxy)*, not as spatial CV, and cite the random-split
gap as a lower bound on the leakage effect. Since the Atlas coverage layer is gone, this is the
strongest available option — but if you ever recover tile centroids, this is the one cell to replace.

### Buffer
`BLOCK_BUFFER=2` base tiles adjacent to each held-out block are dropped from **train only** (never
from val), so the fold boundary is not two neighbouring tiles sitting on opposite sides of the split.
Costs ~1% of training data.

### Grouping
Both splitters operate on the 408 base ids; `expand()` then pulls in all three variants. A base tile
and its flips can never straddle a fold. The permutation in `random_split` is seeded from `cfg.SEED`
alone, so it is **the same partition the U-Net used** — the leakage gap compares like with like.

In [ ]:
def block_split(base_ids: List[int], fold: int, n_folds: int, buffer: int):
    n = len(base_ids)
    edges = [round(i * n / n_folds) for i in range(n_folds + 1)]
    lo, hi = edges[fold], edges[fold + 1]
    val = base_ids[lo:hi]
    train = base_ids[:max(0, lo - buffer)] + base_ids[min(n, hi + buffer):]
    return train, val

def random_split(base_ids: List[int], fold: int, n_folds: int, seed: int):
    ids = list(base_ids)
    random.Random(seed).shuffle(ids)                 # same permutation for all folds
    edges = [round(i * len(ids) / n_folds) for i in range(n_folds + 1)]
    val = ids[edges[fold]:edges[fold + 1]]
    train = ids[:edges[fold]] + ids[edges[fold + 1]:]
    return sorted(train), sorted(val)

if cfg.SPLIT_MODE == "block":
    tr_ids, va_ids = block_split(BASE_IDS, cfg.FOLD, cfg.N_FOLDS, cfg.BLOCK_BUFFER)
elif cfg.SPLIT_MODE == "random":
    tr_ids, va_ids = random_split(BASE_IDS, cfg.FOLD, cfg.N_FOLDS, cfg.SEED)
else:
    raise ValueError(cfg.SPLIT_MODE)

def expand(ids): return [r for i in ids for r in groups[i]]
tr_recs, va_recs = expand(tr_ids), expand(va_ids)

assert not (set(tr_ids) & set(va_ids)), "base-tile leakage between train and val"
print(f"{cfg.SPLIT_MODE} fold {cfg.FOLD}: {len(tr_ids)} train tiles / {len(va_ids)} val tiles "
      f"({len(BASE_IDS)-len(tr_ids)-len(va_ids)} buffered out)")
print(f"  -> {len(tr_recs)} train images / {len(va_recs)} val images")
print(f"  val id range: {min(va_ids)}..{max(va_ids)}")

## 9. Effective-number class weights — computed on the train split only

Weights come from the **training fold alone**. Computing them once over the full dataset would leak
val class priors into the loss; small effect, but exactly the kind of thing a Q1 reviewer flags.

### Important: beta must scale with the counting unit
Effective number (Cui et al. 2019) is `E_c = (1 - beta^n_c) / (1 - beta)`, `w_c = 1/E_c`. The paper
applies it to **per-class image counts** (hundreds to thousands), where `beta=0.9999` is sensible
because `1/(1-beta) = 1e4` is the same order as `n_c`.

Here `n_c` is a **pixel count** (~1e6 to 1e9). At that scale `beta^n_c` underflows to 0 for *every*
class, so `E_c = 1/(1-beta)` for all c and the weights come out **exactly uniform** — i.e. plain
unweighted CE, silently. The cell verifies this and auto-selects `beta = 1 - 1/N_total` instead,
which puts the saturation point at the actual data scale and recovers a meaningful ~50x spread
before capping.

`EN_BETA_AUTO = True` here, matching `unet.ipynb`. This setting must be identical across all seven
baselines — do not mix.

In [ ]:
EN_BETA_AUTO = True     # False = reproduce the legacy (saturated -> uniform) behaviour

def effective_number_weights(recs, cap, n_classes, beta_fixed, auto):
    n = np.zeros(n_classes, dtype=np.float64)
    for _, _, iname, _ in recs:
        n += PIX_COUNTS[iname][:n_classes]
    beta = (1.0 - 1.0 / n.sum()) if auto else beta_fixed
    eff = (1.0 - np.power(beta, n)) / (1.0 - beta)
    w = 1.0 / np.maximum(eff, 1e-12)
    w = w / w.mean()
    w = np.minimum(w, cap)
    return w / w.mean(), n, beta

CLASS_W, TRAIN_PX, BETA_USED = effective_number_weights(
    tr_recs, cfg.WEIGHT_CAP, cfg.N_CLASSES, cfg.EN_BETA, EN_BETA_AUTO)

print(f"total train pixels: {TRAIN_PX.sum():.3e} | beta used: {BETA_USED:.12f} "
      f"| saturation point 1/(1-beta) = {1/(1-BETA_USED):.3e}")
print(f"{'class':<14}{'train px %':>12}{'weight':>10}")
for i, nm in enumerate(cfg.CLASS_NAMES):
    print(f"{nm:<14}{TRAIN_PX[i]/TRAIN_PX.sum()*100:11.2f}%{CLASS_W[i]:10.3f}")
print(f"weight ratio max/min: {CLASS_W.max()/CLASS_W.min():.1f}")

if CLASS_W.max() / CLASS_W.min() < 1.05:
    print("\nWARNING: weights are effectively uniform -- beta has saturated. "
          "This is unweighted CE. Set EN_BETA_AUTO=True unless you are deliberately "
          "matching an earlier saturated run.")

CLASS_W_T = torch.tensor(CLASS_W, dtype=torch.float32).cuda()

## 10. Dataset and DataLoaders

**No hflip/vflip** — those are already materialised in the 1224 files. Adding them would produce
duplicate views inside a single epoch.

On-the-fly augmentation is **rot90 (k in 0..3) + mild brightness/contrast jitter**. Rationale:
combined with the baked-in flips this completes the dihedral group D4, which is the natural symmetry
group for nadir overhead imagery, and it introduces no new interpolation (rot90 is exact — no
resampling, so 10-20 px roads stay crisp). Scale jitter is deliberately excluded: it changes effective
GSD, which would make the thin-structure results harder to interpret. Since you are applying this
across all seven baselines, it stays a controlled constant rather than a confound.

Augmentation is seeded per (epoch, index) so the run is reproducible and workers never collide.
Byte-for-byte identical to the U-Net dataset class — if you ever change augmentation, change it
in all seven notebooks in the same commit.

In [ ]:
IMNET_MEAN = np.array([0.485, 0.456, 0.406], np.float32)
IMNET_STD  = np.array([0.229, 0.224, 0.225], np.float32)

class SatSeg(Dataset):
    def __init__(self, recs, train: bool):
        self.recs, self.train, self.epoch = recs, train, 0

    def __len__(self): return len(self.recs)

    def _read(self, iname, mname):
        if cfg.PRELOAD_RAM:
            img, lab = RAM[iname]
            return img.copy(), lab.copy()
        img = cv2.imread(str(CACHE / "images" / iname), cv2.IMREAD_COLOR)
        lab = cv2.imread(str(CACHE / "labels" / mname), cv2.IMREAD_GRAYSCALE)
        return img, lab

    def __getitem__(self, i):
        _, _, iname, mname = self.recs[i]
        img, lab = self._read(iname, mname)

        if self.train:
            rs = np.random.RandomState((cfg.SEED * 100003 + self.epoch * 9973 + i) % (2**31))
            if cfg.AUG_ROT90:
                k = rs.randint(4)
                if k:
                    img = np.ascontiguousarray(np.rot90(img, k))
                    lab = np.ascontiguousarray(np.rot90(lab, k))
            if cfg.AUG_PHOTOMETRIC:
                a = 1.0 + rs.uniform(-cfg.AUG_CONTRAST, cfg.AUG_CONTRAST)        # contrast
                b = 255.0 * rs.uniform(-cfg.AUG_BRIGHTNESS, cfg.AUG_BRIGHTNESS)  # brightness
                img = np.clip(img.astype(np.float32) * a + b, 0, 255).astype(np.uint8)

        x = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        x = (x - IMNET_MEAN) / IMNET_STD
        x = torch.from_numpy(x.transpose(2, 0, 1))
        y = torch.from_numpy(lab.astype(np.int64))
        return x, y

train_ds, val_ds = SatSeg(tr_recs, True), SatSeg(va_recs, False)
train_ld = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True, drop_last=True,
                      num_workers=cfg.NUM_WORKERS, pin_memory=True, persistent_workers=True,
                      worker_init_fn=worker_init_fn, generator=g)
val_ld   = DataLoader(val_ds, batch_size=cfg.BATCH_SIZE, shuffle=False,
                      num_workers=cfg.NUM_WORKERS, pin_memory=True, persistent_workers=True,
                      worker_init_fn=worker_init_fn)
xb, yb = next(iter(train_ld))
print(xb.shape, xb.dtype, yb.shape, yb.dtype, "| label values:", torch.unique(yb).tolist()[:8])

## 11. Loss — weighted CE + Lovasz-Softmax

Lovasz-Softmax is the convex surrogate of the Jaccard loss (Berman et al. 2018) and is what actually
moves mIoU on thin classes; weighted CE keeps early optimization stable. `lambda=0.75` sits in the
0.5-1.0 range you specified.

Two implementation details that matter:
- The whole loss is computed in **fp32 outside autocast**. Lovasz sorts per-pixel errors and its
  gradient is a cumulative-sum difference; in fp16 that accumulates visible error and can produce NaNs.
- Ignore pixels are filtered *before* flattening, so 255 never reaches either term.

**New vs the U-Net run: `label_smoothing=0.05` on the CE term.** Your U-Net val loss climbed from
1.3 to 2.8 while val mIoU was still improving — the signature of a model driving its softmax toward
one-hot on pixels it gets wrong, which costs unbounded CE and buys nothing. Smoothing caps the
per-pixel reward for extra confidence, which both damps that climb and acts as a genuine regularizer.
`nn.CrossEntropyLoss` applies it correctly alongside `weight` and `ignore_index` on torch >= 1.11.
Two consequences to expect: **val loss now has a non-zero floor** (roughly 0.15-0.25 higher than
before), so compare curve *shape*, not absolute values, against the U-Net run; and the Lovasz term
is untouched, so mIoU optimization is unchanged.

In [ ]:
def lovasz_grad(gt_sorted):
    p = len(gt_sorted); gts = gt_sorted.sum()
    inter = gts - gt_sorted.float().cumsum(0)
    union = gts + (1 - gt_sorted).float().cumsum(0)
    jac = 1.0 - inter / union
    if p > 1: jac[1:p] = jac[1:p] - jac[0:-1]
    return jac

def lovasz_softmax_flat(probas, labels, n_classes):
    if probas.numel() == 0: return probas.sum() * 0.0
    losses = []
    for c in range(n_classes):
        fg = (labels == c).float()
        if fg.sum() == 0: continue                     # "present" classes only
        err = (fg - probas[:, c]).abs()
        err_sorted, perm = torch.sort(err, 0, descending=True)
        losses.append(torch.dot(err_sorted, lovasz_grad(fg[perm])))
    return torch.stack(losses).mean() if losses else probas.sum() * 0.0

class ComboLoss(nn.Module):
    def __init__(self, weights, lam, ignore, n_classes, smooth=0.0):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(weight=weights, ignore_index=ignore, label_smoothing=smooth)
        self.lam, self.ignore, self.nc = lam, ignore, n_classes

    def forward(self, logits, target):
        logits = logits.float()                        # fp32 regardless of autocast
        ce = self.ce(logits, target)
        B, C, H, W = logits.shape
        p = logits.permute(0, 2, 3, 1).reshape(-1, C).softmax(1)
        t = target.reshape(-1)
        keep = t != self.ignore
        lv = lovasz_softmax_flat(p[keep], t[keep], self.nc)
        return ce + self.lam * lv, ce.detach(), lv.detach()

criterion = ComboLoss(CLASS_W_T, cfg.LAMBDA_LOVASZ, cfg.IGNORE_INDEX,
                      cfg.N_CLASSES, cfg.LABEL_SMOOTHING).cuda()
print("loss ready | lambda =", cfg.LAMBDA_LOVASZ, "| label_smoothing =", cfg.LABEL_SMOOTHING)

## 12. Model, optimizer, scheduler

### The resolution trap
`SegformerForSemanticSegmentation` returns logits at **H/4 x W/4 = 128x128**, not at input
resolution — the all-MLP head fuses the four MiT stages at 1/4 scale and stops there. Feeding those
straight into the loss would compare 128x128 logits against 512x512 labels and throw a shape error;
worse, downsampling the *labels* to match (the "obvious" fix) would destroy exactly the 10-20 px road
structures this benchmark is about. **Every forward pass in this notebook goes through
`forward_logits()`**, which bilinearly upsamples to the input size first. That is the mmsegmentation
convention and it keeps the loss, the metrics, and the U-Net comparison all at native 512.

### Parameter groups
| Group | LR | Weight decay |
|---|---|---|
| encoder (`model.segformer`), weights | 6e-6 | 0.01 |
| encoder, LayerNorm gains + biases | 6e-6 | 0 |
| decode head, weights | 6e-5 | 0.01 |
| decode head, LayerNorm gains + biases | 6e-5 | 0 |

Excluding norm gains and biases from decay is standard for transformers and non-optional in
practice: decaying a LayerNorm gain toward zero fights the normalization itself. `wd=0.01` is the
SegFormer paper's value — 100x the U-Net's 1e-4, and the main per-architecture contribution to the
overfitting fix.

### Regularization inside the architecture
`drop_path_rate=0.1` (stochastic depth, linearly scaled across MiT blocks by HF) and
`classifier_dropout_prob=0.1` are the published MiT-B0 defaults. If the val curves still diverge the
way the U-Net's did, **raise `DROP_PATH_RATE` to 0.2 first** — it is the cheapest and least
protocol-invasive knob you have, and it is per-architecture so it does not oblige a U-Net re-run.

### Schedule
500 linear warmup steps, then cosine decay to zero at **`COS_HORIZON_EPOCHS=120`** rather than at
`MAX_EPOCHS=200`. In the U-Net run the cosine was still at ~40% of peak LR when training stopped
near epoch 100, so the model spent its entire late phase at a learning rate high enough to keep
memorizing. Annealing to zero inside the horizon where early stopping actually fires gives the usual
end-of-cosine generalization bump. Epochs past 120 run at LR ~0 and are harmless — patience will
have fired long before.

### EMA
An exponential moving average of the weights (decay 0.999, warmed up so the first few hundred steps
are not dominated by the initialization) is what gets validated and checkpointed. EMA is the
cheapest variance reducer available for a small model on 272 scenes and typically buys +0.5-1.5 mIoU
with no extra compute. **This is a shared-protocol change** — apply it to every baseline or none.

In [ ]:
from transformers import SegformerForSemanticSegmentation, SegformerConfig

def build_model():
    return SegformerForSemanticSegmentation.from_pretrained(
        cfg.HF_MODEL_ID,
        num_labels=cfg.N_CLASSES,
        ignore_mismatched_sizes=True,           # ADE20K head is 150-way; ours is 6-way
        drop_path_rate=cfg.DROP_PATH_RATE,
        classifier_dropout_prob=cfg.CLASSIFIER_DROPOUT,
        hidden_dropout_prob=cfg.HIDDEN_DROPOUT,
        attention_probs_dropout_prob=cfg.ATTENTION_DROPOUT,
    )

def forward_logits(m, x):
    # SegFormer emits H/4 logits; upsample to input size so loss and metrics stay at native 512.
    lo = m(pixel_values=x).logits
    return F.interpolate(lo, size=x.shape[-2:], mode="bilinear", align_corners=False)

model = build_model().cuda()

def param_groups(m):
    def split(mod):
        dec, nod = [], []
        for n, p in mod.named_parameters():
            if not p.requires_grad: continue
            (nod if (p.ndim <= 1 or n.endswith(".bias")) else dec).append(p)
        return dec, nod
    e_d, e_n = split(m.segformer)
    h_d, h_n = split(m.decode_head)
    wd = cfg.WEIGHT_DECAY
    nwd = 0.0 if cfg.NO_DECAY_ON_NORM_BIAS else wd
    return [
        {"params": e_d, "lr": cfg.LR_ENCODER, "weight_decay": wd,  "name": "enc_decay"},
        {"params": e_n, "lr": cfg.LR_ENCODER, "weight_decay": nwd, "name": "enc_nodecay"},
        {"params": h_d, "lr": cfg.LR_DECODER, "weight_decay": wd,  "name": "head_decay"},
        {"params": h_n, "lr": cfg.LR_DECODER, "weight_decay": nwd, "name": "head_nodecay"},
    ]

optimizer = torch.optim.AdamW(param_groups(model), weight_decay=cfg.WEIGHT_DECAY)
LR_IDX_ENC, LR_IDX_DEC = 0, 2                  # indices used for logging

steps_per_epoch = len(train_ld) // cfg.GRAD_ACCUM
total_steps = steps_per_epoch * cfg.COS_HORIZON_EPOCHS      # cosine horizon, not MAX_EPOCHS

def lr_lambda(step):
    if step < cfg.WARMUP_STEPS:
        return (step + 1) / cfg.WARMUP_STEPS
    prog = (step - cfg.WARMUP_STEPS) / max(1, total_steps - cfg.WARMUP_STEPS)
    return 0.5 * (1.0 + math.cos(math.pi * min(1.0, prog)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = torch.cuda.amp.GradScaler(enabled=cfg.AMP)

class ModelEMA:
    def __init__(self, m, decay):
        self.ema = copy.deepcopy(m).eval()
        for p in self.ema.parameters(): p.requires_grad_(False)
        self.decay = decay
    @torch.no_grad()
    def update(self, m, step):
        # ramp the decay in, otherwise the average stays pinned to the init for thousands of steps
        d = min(self.decay, (1.0 + step) / (10.0 + step))
        msd = m.state_dict()
        for k, v in self.ema.state_dict().items():
            if v.dtype.is_floating_point: v.mul_(d).add_(msd[k].detach(), alpha=1.0 - d)
            else: v.copy_(msd[k])

ema = ModelEMA(model, cfg.EMA_DECAY) if cfg.USE_EMA else None
eval_model = ema.ema if cfg.USE_EMA else model

n_par = sum(p.numel() for p in model.parameters())
with torch.no_grad():
    probe = forward_logits(model, torch.zeros(1, 3, cfg.IMG_SIZE, cfg.IMG_SIZE, device="cuda"))
assert probe.shape[-2:] == (cfg.IMG_SIZE, cfg.IMG_SIZE), probe.shape
print(f"params: {n_par/1e6:.2f} M | logits after upsample: {tuple(probe.shape)}")
print(f"steps/epoch: {steps_per_epoch} | cosine horizon: {total_steps} steps "
      f"({cfg.COS_HORIZON_EPOCHS} epochs) | EMA: {cfg.USE_EMA}")

## 13. Metrics module

Identical to the U-Net notebook — the metric code is protocol, so it is copied verbatim rather than
rewritten. Everything accumulates into a single 6x6 confusion matrix built on-GPU with `bincount`
(fast enough to run every epoch). Ignore pixels are dropped before accumulation, so 255 never enters
any metric.

- **IoU / F1 / mIoU / mF1** — macro over the 6 classes
- **OA**, **frequency-weighted IoU**
- **Cohen's kappa** — from the same matrix
- **Producer's accuracy** = recall (row-normalized), **User's accuracy** = precision (column-normalized).
  These are the remote-sensing names for the same quantities; reporting both plus the normalized
  matrix is what reviewers in this field expect.
- **clDice** (Shit et al. 2021) for **Road** and **River** only — a topology-aware score that rewards
  connectivity rather than area overlap, which is the right lens for 10-20 px linear features.
  Skeletonization is CPU-bound, so it runs at final evaluation only, not every epoch.

In [ ]:
from skimage.morphology import skeletonize

class ConfMat:
    def __init__(self, n): self.n = n; self.m = torch.zeros(n, n, dtype=torch.int64, device="cuda")
    def update(self, pred, tgt):
        k = tgt != cfg.IGNORE_INDEX
        if k.sum() == 0: return
        self.m += torch.bincount(self.n * tgt[k].long() + pred[k].long(),
                                 minlength=self.n ** 2).reshape(self.n, self.n)
    def numpy(self): return self.m.cpu().numpy().astype(np.float64)

def metrics_from_cm(cm, names):
    tp = np.diag(cm); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp; tot = cm.sum()
    iou = tp / np.maximum(tp + fp + fn, 1e-12)
    f1  = 2 * tp / np.maximum(2 * tp + fp + fn, 1e-12)
    prod = tp / np.maximum(cm.sum(1), 1e-12)      # recall
    user = tp / np.maximum(cm.sum(0), 1e-12)      # precision
    freq = cm.sum(1) / max(tot, 1e-12)
    po = tp.sum() / max(tot, 1e-12)
    pe = (cm.sum(0) * cm.sum(1)).sum() / max(tot ** 2, 1e-12)
    return {
        "mIoU": float(np.nanmean(iou)), "mF1": float(np.nanmean(f1)),
        "OA": float(po), "FWIoU": float((freq * iou).sum()),
        "kappa": float((po - pe) / max(1 - pe, 1e-12)),
        "per_class": {n: {"IoU": float(iou[i]), "F1": float(f1[i]),
                          "producer_acc": float(prod[i]), "user_acc": float(user[i]),
                          "support_px": int(cm.sum(1)[i])} for i, n in enumerate(names)},
        "confusion_matrix_raw": cm.astype(np.int64).tolist(),
        "confusion_matrix_norm": (cm / np.maximum(cm.sum(1, keepdims=True), 1e-12)).tolist(),
    }

def cldice_pair(pred_bin, gt_bin):
    if gt_bin.sum() == 0 and pred_bin.sum() == 0: return None
    sp = skeletonize(pred_bin.astype(bool)); sg = skeletonize(gt_bin.astype(bool))
    tprec = (sp & gt_bin.astype(bool)).sum() / max(sp.sum(), 1)     # topology precision
    tsens = (sg & pred_bin.astype(bool)).sum() / max(sg.sum(), 1)   # topology sensitivity
    if tprec + tsens == 0: return 0.0
    return float(2 * tprec * tsens / (tprec + tsens))

## 14. Efficiency metrics

Params, GFLOPs at 512x512, peak training VRAM, wall-clock, and inference FPS. FPS is measured at
batch 1 with 20 warmup iterations and CUDA synchronization — without the warmup you are timing cuDNN
autotuning, and on a shared Kaggle T4 you should treat FPS as indicative rather than exact.

Two SegFormer-specific wrinkles:
- `ptflops` needs a module whose `forward` returns a tensor, so the HF model is wrapped. The wrapper
  includes the H/4 -> 512 upsample, matching what actually runs at inference.
- **`ptflops` hooks `nn.Module`s and therefore misses the functional attention matmuls** (QK^T and
  AV). For B0 at 512 with sequence reduction that is a small fraction, but it means transformer rows
  in your efficiency table are slightly *under*-counted relative to CNN rows. Say so in the caption,
  or re-measure the whole table with `fvcore` — the important thing is that all seven rows use the
  same tool. Published reference: SegFormer-B0 is ~3.8 M params / ~8.4 GFLOPs at 512x512.

In [ ]:
class _FlopsWrapper(nn.Module):
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, x): return forward_logits(self.m, x)

def efficiency_report(m):
    from ptflops import get_model_complexity_info
    m.eval()
    w = _FlopsWrapper(m)
    with torch.cuda.amp.autocast(enabled=False):
        macs, params = get_model_complexity_info(
            w, (3, cfg.IMG_SIZE, cfg.IMG_SIZE),
            as_strings=False, print_per_layer_stat=False, verbose=False)
    x = torch.randn(1, 3, cfg.IMG_SIZE, cfg.IMG_SIZE, device="cuda")
    with torch.no_grad(), torch.cuda.amp.autocast(enabled=cfg.AMP):
        for _ in range(20): forward_logits(m, x)
        torch.cuda.synchronize(); t0 = time.time()
        for _ in range(50): forward_logits(m, x)
        torch.cuda.synchronize()
    return {"params_M": params / 1e6,
            "GFLOPs_512": macs * 2 / 1e9,           # ptflops returns MACs; FLOPs = 2 x MACs
            "inference_fps_bs1": 50 / (time.time() - t0)}

## 15. Hugging Face persistence and resume

Each fold gets its **own repo** (`geonusaf-segformer-b0-{mode}-fold{k}`), matching the U-Net pattern,
so any single fold can be pulled independently for inference later and nothing is overwritten across
folds. The aggregation cell in section 21 depends on this layout.

Two artifacts per repo:
- `best.pt` — weights (EMA weights when `USE_EMA`) + config + metrics, pushed whenever val mIoU
  improves. This is what the inference script downloads.
- `last.pt` — full state incl. optimizer/scaler/scheduler/EMA, pushed every `PUSH_LAST_EVERY`
  epochs. This is what a new session resumes from.

B0 is small enough that both files are tens of MB rather than the U-Net's ~130/390 MB, so the upload
cost per push is negligible here — `PUSH_LAST_EVERY=10` is kept only for consistency.

In [ ]:
from huggingface_hub import HfApi, create_repo, hf_hub_download
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    HF_TOKEN = os.environ.get("HF_TOKEN"); print("secret fallback:", e)

api = HfApi(token=HF_TOKEN)
create_repo(HF_REPO, token=HF_TOKEN, exist_ok=True, private=False)

def hf_push(local_path, name, msg):
    api.upload_file(path_or_fileobj=local_path, path_in_repo=name,
                    repo_id=HF_REPO, token=HF_TOKEN, commit_message=msg)

def hf_try_download(name):
    try: return hf_hub_download(HF_REPO, name, token=HF_TOKEN)
    except Exception: return None

def push_model_card(best_metrics, epoch):
    pc = best_metrics.get("per_class", {})
    L = []
    L.append("---")
    L.append("tags: [semantic-segmentation, remote-sensing, segformer, mit-b0, geonusaf]")
    L.append("library_name: transformers")
    L.append("---")
    L.append("")
    L.append(f"# GeoNUSAF - SegFormer-B0 - {cfg.SPLIT_MODE} split, fold {cfg.FOLD}")
    L.append("")
    L.append("Kathmandu Valley land-use segmentation, 6 classes, ignore_index=255.")
    L.append("")
    L.append("| field | value |")
    L.append("|---|---|")
    L.append(f"| architecture | SegformerForSemanticSegmentation, {cfg.HF_MODEL_ID} |")
    L.append(f"| split mode | {cfg.SPLIT_MODE} |")
    L.append(f"| fold | {cfg.FOLD} of {cfg.N_FOLDS} |")
    L.append(f"| seed | {cfg.SEED} |")
    L.append("| input | 512x512, ImageNet norm, effective GSD 0.586 m/px |")
    L.append(f"| classes | {', '.join(cfg.CLASS_NAMES)} |")
    L.append(f"| lr (head/encoder) | {cfg.LR_DECODER} / {cfg.LR_ENCODER} |")
    L.append(f"| regularization | wd {cfg.WEIGHT_DECAY}, drop_path {cfg.DROP_PATH_RATE}, "
             f"smooth {cfg.LABEL_SMOOTHING}, EMA {cfg.USE_EMA} |")
    L.append(f"| best epoch | {epoch} |")
    L.append(f"| val mIoU | {best_metrics.get('mIoU', float('nan')):.4f} |")
    L.append(f"| val mF1 | {best_metrics.get('mF1', float('nan')):.4f} |")
    L.append(f"| val OA | {best_metrics.get('OA', float('nan')):.4f} |")
    L.append(f"| val kappa | {best_metrics.get('kappa', float('nan')):.4f} |")
    L.append("")
    L.append("## Per-class (validation)")
    L.append("| class | IoU | F1 |")
    L.append("|---|---|---|")
    for k, v in pc.items():
        L.append(f"| {k} | {v['IoU']:.4f} | {v['F1']:.4f} |")
    L.append("")
    L.append("Checkpoint `best.pt` holds `model_state` (EMA weights when EMA is on), the run `cfg`, "
             "the full HF `model_config`, and `metrics`.")
    card = "\n".join(L)
    p = f"{cfg.OUT_DIR}/README.md"; open(p, "w").write(card)
    hf_push(p, "README.md", f"card epoch {epoch}")

print("HF ready:", HF_REPO)

## 16. Weights & Biases

Matches the logging you have on the other baselines. `resume="allow"` with a deterministic run id
means a resumed session continues the same W&B run rather than starting a second one.

One extra series here: **`overfit_gap` = val_loss - train_loss**, logged every epoch. That is the
quantity your two U-Net screenshots were really showing, and having it as a single scalar makes the
seven-baseline comparison of overfitting behaviour a one-panel plot instead of an eyeball exercise.

In [ ]:
import wandb
try:
    wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))
    WANDB = True
except Exception as e:
    print("wandb disabled:", e); WANDB = False

if WANDB:
    wandb.init(project=cfg.WANDB_PROJECT, name=RUN_NAME, id=RUN_NAME,
               resume="allow", config=asdict(cfg))

## 17. Train / validate loops

Notes on the mechanics:
- `scaler.unscale_(optimizer)` runs **before** `clip_grad_norm_`, otherwise you would be clipping
  loss-scaled gradients and the 1.0 threshold would mean nothing. `clip_grad_norm_` returns the
  pre-clip norm, which is what gets logged — your direct check against the 6e5 explosion from the
  earlier inverse-frequency-weighted SegFormer run.
- The EMA updates **after** `scaler.step`, and only on steps the optimizer actually took — a step
  skipped by the GradScaler on an inf/NaN check must not advance the average toward unused weights.
- Validation runs on the EMA weights at full 512x512 with no TTA. Keep it that way across all
  baselines; TTA belongs in a separate row of the table, not in the protocol.
- Early stopping tracks **val mIoU**, not val loss — the loss is class-weighted, now smoothed, and
  moves for reasons mIoU does not care about. This is exactly why your U-Net's rising val loss was
  not by itself proof the run had gone bad.

In [ ]:
def train_one_epoch(epoch):
    model.train(); train_ds.epoch = epoch
    tot, n, gn_acc = 0.0, 0, []
    optimizer.zero_grad(set_to_none=True)
    for it, (x, y) in enumerate(train_ld):
        x = x.cuda(non_blocking=True); y = y.cuda(non_blocking=True)
        with torch.cuda.amp.autocast(enabled=cfg.AMP):
            logits = forward_logits(model, x)
        loss, ce, lv = criterion(logits, y)
        scaler.scale(loss / cfg.GRAD_ACCUM).backward()

        if (it + 1) % cfg.GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)                                  # unscale BEFORE clipping
            gn = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.CLIP_NORM)
            gn_acc.append(float(gn))
            prev_scale = scaler.get_scale()
            scaler.step(optimizer); scaler.update()
            optimizer.zero_grad(set_to_none=True); scheduler.step()
            if ema is not None and scaler.get_scale() >= prev_scale:    # skip EMA on skipped steps
                ema.update(model, scheduler.last_epoch)

        tot += float(loss) * x.size(0); n += x.size(0)
        if it % 20 == 0:
            print(f"  e{epoch} it{it}/{len(train_ld)} loss {float(loss):.4f} "
                  f"(ce {float(ce):.4f} lov {float(lv):.4f}) gnorm {gn_acc[-1] if gn_acc else 0:.3f}",
                  flush=True)
    return tot / max(n, 1), float(np.mean(gn_acc)), float(np.max(gn_acc))

@torch.no_grad()
def validate(m=None):
    m = m if m is not None else eval_model
    m.eval(); cm = ConfMat(cfg.N_CLASSES); tot, n = 0.0, 0
    for x, y in val_ld:
        x = x.cuda(non_blocking=True); y = y.cuda(non_blocking=True)
        with torch.cuda.amp.autocast(enabled=cfg.AMP):
            logits = forward_logits(m, x)
        loss, _, _ = criterion(logits, y)
        cm.update(logits.float().argmax(1), y)
        tot += float(loss) * x.size(0); n += x.size(0)
    return tot / max(n, 1), metrics_from_cm(cm.numpy(), cfg.CLASS_NAMES)

## 18. Main loop with automatic resume

On start this pulls `last.pt` from the fold's HF repo if it exists and continues from that epoch.
So the answer to "what if it exceeds 12 hours" is: it stops, you start a new session, run the
notebook top to bottom unchanged, and it picks up where it left off.

**Expected cost.** ~816 train images, batch 16, 51 steps/epoch; 408 val images, 26 batches. B0 is
the lightest model in your lineup (~8.4 GFLOPs at 512 vs the U-Net-R50's ~100), so with the cache in
RAM expect roughly **25-50 s/epoch** including validation and the EMA copy. With the cosine horizon
at 120 epochs, early stopping should fire somewhere in the 60-120 range: **~1.0-1.7 h per fold**, and
**~6-10 h for all six runs** (3 block + 3 random). That is a third of your 30 GPU-h weekly quota —
comfortable, and it leaves room to re-run a fold if the LR verification below says 6e-5 was wrong.

**VRAM.** B0 at batch 16, 512x512, AMP lands around **5-7 GB of 16 GB**. An OOM here would be
surprising; if one happens it will be from the 128x128x6 logits upsample buffer, and
`BATCH_SIZE=8, GRAD_ACCUM=2` keeps the effective batch at 16 and the protocol unchanged.

**LR verification (do this on the first fold).** Compare val mIoU at epochs 5-10 against your earlier
3e-5 exploratory run. If 6e-5 is clearly ahead or level, keep it and use it for B2 as well. If it is
behind *and* grad norms are riding the 1.0 clip constantly, drop to 3e-5 — but then re-run every
SegFormer fold at that value. Do not let two SegFormer rows in the table use different LRs.

In [ ]:
ck = hf_try_download("last.pt")
start_epoch, best_miou, best_epoch, bad, history = 0, -1.0, -1, 0, []
if ck:
    st = torch.load(ck, map_location="cuda")
    model.load_state_dict(st["model_state"]); optimizer.load_state_dict(st["optim_state"])
    scheduler.load_state_dict(st["sched_state"]); scaler.load_state_dict(st["scaler_state"])
    if ema is not None and st.get("ema_state") is not None:
        ema.ema.load_state_dict(st["ema_state"])
    start_epoch = st["epoch"] + 1; best_miou = st["best_miou"]; best_epoch = st["best_epoch"]
    bad = st["bad"]; history = st["history"]
    print(f"RESUMED from epoch {st['epoch']} (best mIoU {best_miou:.4f} @ e{best_epoch})")
else:
    print("fresh start")

torch.cuda.reset_peak_memory_stats()
t_start, best_metrics = time.time(), {}

for epoch in range(start_epoch, cfg.MAX_EPOCHS):
    te = time.time()
    tr_loss, gn_mean, gn_max = train_one_epoch(epoch)
    va_loss, m = validate()
    peak_gb = torch.cuda.max_memory_allocated() / 1e9
    row = {"epoch": epoch, "train_loss": tr_loss, "val_loss": va_loss,
           "overfit_gap": va_loss - tr_loss, "val_mIoU": m["mIoU"],
           "val_mF1": m["mF1"], "val_OA": m["OA"], "val_kappa": m["kappa"],
           "grad_norm_mean": gn_mean, "grad_norm_max": gn_max,
           "lr_enc": scheduler.get_last_lr()[LR_IDX_ENC],
           "lr_dec": scheduler.get_last_lr()[LR_IDX_DEC], "peak_vram_gb": peak_gb,
           "epoch_sec": time.time() - te,
           **{f"IoU/{k}": v["IoU"] for k, v in m["per_class"].items()}}
    history.append(row)
    if WANDB: wandb.log(row, step=epoch)
    print(f"[e{epoch}] train {tr_loss:.4f} | val {va_loss:.4f} | gap {row['overfit_gap']:+.3f} | "
          f"mIoU {m['mIoU']:.4f} | gnorm mu{gn_mean:.2f}/max{gn_max:.2f} | "
          f"{row['epoch_sec']:.0f}s | vram {peak_gb:.1f}GB")

    if m["mIoU"] > best_miou:
        best_miou, best_epoch, bad, best_metrics = m["mIoU"], epoch, 0, m
        p = f"{cfg.OUT_DIR}/best.pt"
        torch.save({"model_state": eval_model.state_dict(),      # EMA weights when EMA is on
                    "model_config": model.config.to_dict(),
                    "cfg": asdict(cfg), "metrics": m, "epoch": epoch}, p)
        hf_push(p, "best.pt", f"best mIoU {best_miou:.4f} @ e{epoch}")
        push_model_card(best_metrics, epoch)
        print(f"  -> new best {best_miou:.4f}, pushed")
    else:
        bad += 1

    if (epoch % cfg.PUSH_LAST_EVERY == 0) or (bad >= cfg.PATIENCE) or (epoch == cfg.MAX_EPOCHS - 1):
        p = f"{cfg.OUT_DIR}/last.pt"
        torch.save({"model_state": model.state_dict(), "optim_state": optimizer.state_dict(),
                    "sched_state": scheduler.state_dict(), "scaler_state": scaler.state_dict(),
                    "ema_state": (ema.ema.state_dict() if ema is not None else None),
                    "epoch": epoch, "best_miou": best_miou, "best_epoch": best_epoch,
                    "bad": bad, "history": history}, p)
        hf_push(p, "last.pt", f"state e{epoch}")

    if bad >= cfg.PATIENCE:
        print(f"EARLY STOP at epoch {epoch} (best {best_miou:.4f} @ e{best_epoch})"); break

wall_h = (time.time() - t_start) / 3600
print(f"training done | best mIoU {best_miou:.4f} @ epoch {best_epoch} | this session {wall_h:.2f} h")

## 19. Final evaluation of the best checkpoint

Reloads `best.pt` (EMA weights) into a fresh model and computes the **full** metric suite on the
held-out fold, including clDice for Road and River (skeletonization per image, CPU) and the
efficiency block. clDice is averaged over images in which the class is present in GT or prediction;
images where the class is absent from both are skipped rather than scored as 1.0, which would
inflate the number badly given how sparse River is.

In [ ]:
_ck = torch.load(f"{cfg.OUT_DIR}/best.pt", map_location="cuda")
best_model = build_model().cuda()
best_model.load_state_dict(_ck["model_state"])
best_model.eval()

cm = ConfMat(cfg.N_CLASSES)
cld = {1: [], 2: []}                                   # Road, River
with torch.no_grad():
    for x, y in val_ld:
        x = x.cuda(non_blocking=True); y = y.cuda(non_blocking=True)
        with torch.cuda.amp.autocast(enabled=cfg.AMP):
            logits = forward_logits(best_model, x)
        pred = logits.float().argmax(1)
        cm.update(pred, y)
        pn, yn = pred.cpu().numpy(), y.cpu().numpy()
        valid = yn != cfg.IGNORE_INDEX
        for b in range(pn.shape[0]):
            for c in cld:
                pb = (pn[b] == c) & valid[b]
                gb = (yn[b] == c) & valid[b]
                v = cldice_pair(pb, gb)
                if v is not None: cld[c].append(v)

final = metrics_from_cm(cm.numpy(), cfg.CLASS_NAMES)
final["clDice"] = {cfg.CLASS_NAMES[c]: (float(np.mean(v)) if v else None) for c, v in cld.items()}
final["clDice_n_images"] = {cfg.CLASS_NAMES[c]: len(v) for c, v in cld.items()}
final.update(efficiency_report(best_model))
final["peak_train_vram_gb"] = float(torch.cuda.max_memory_allocated() / 1e9)
final["train_wall_clock_h_this_session"] = wall_h
final["best_epoch"] = best_epoch
final["run"] = {"model": "SegFormer-B0", "split_mode": cfg.SPLIT_MODE,
                "fold": cfg.FOLD, "seed": cfg.SEED}

print(json.dumps({k: v for k, v in final.items()
                  if k not in ("confusion_matrix_raw", "confusion_matrix_norm", "per_class")}, indent=2))
print("\nper class:")
for k, v in final["per_class"].items():
    print(f"  {k:<14} IoU {v['IoU']:.4f}  F1 {v['F1']:.4f}  PA {v['producer_acc']:.4f}  UA {v['user_acc']:.4f}")

## 20. Persist results

Writes `results_{mode}_fold{k}.json` (full, incl. confusion matrices), a flat `.csv` row for the
comparison table, and `history.csv` for the curves — then pushes all three to the fold's HF repo so
they survive the session. The flat row carries the regularization settings as columns, so the
comparison table can never silently mix a smoothed run with an unsmoothed one.

In [ ]:
import csv
tag = f"{cfg.SPLIT_MODE}_fold{cfg.FOLD}"
jp = f"{cfg.OUT_DIR}/results_{tag}.json"
json.dump(final, open(jp, "w"), indent=2)

flat = {"model": "SegFormer-B0", "split_mode": cfg.SPLIT_MODE, "fold": cfg.FOLD, "seed": cfg.SEED,
        "mIoU": final["mIoU"], "mF1": final["mF1"], "OA": final["OA"],
        "FWIoU": final["FWIoU"], "kappa": final["kappa"],
        "params_M": final["params_M"], "GFLOPs_512": final["GFLOPs_512"],
        "fps_bs1": final["inference_fps_bs1"], "peak_vram_gb": final["peak_train_vram_gb"],
        "best_epoch": final["best_epoch"],
        "lr_dec": cfg.LR_DECODER, "weight_decay": cfg.WEIGHT_DECAY,
        "label_smoothing": cfg.LABEL_SMOOTHING, "cos_horizon": cfg.COS_HORIZON_EPOCHS,
        "ema": cfg.USE_EMA, "drop_path": cfg.DROP_PATH_RATE}
for k, v in final["per_class"].items():
    flat[f"IoU_{k}"] = v["IoU"]; flat[f"F1_{k}"] = v["F1"]
for k, v in final["clDice"].items():
    flat[f"clDice_{k}"] = v

cp = f"{cfg.OUT_DIR}/results_{tag}.csv"
with open(cp, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(flat)); w.writeheader(); w.writerow(flat)

hp = f"{cfg.OUT_DIR}/history_{tag}.csv"
with open(hp, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(history[0])); w.writeheader(); w.writerows(history)

for p, n in [(jp, f"results_{tag}.json"), (cp, f"results_{tag}.csv"), (hp, f"history_{tag}.csv")]:
    hf_push(p, n, "results")
if WANDB: wandb.finish()
print("results pushed to", HF_REPO)

## 21. Aggregation across folds — run after all folds finish

Pulls each fold's CSV from its own HF repo and reports mean +/- std per split mode. The
**block-mode mIoU minus random-mode mIoU is the leakage gap** — the headline number justifying why
the block protocol is the one you report. Expect random to score higher; if the gap is near zero,
that is itself a finding worth stating (it would suggest the export-order proxy is not capturing
much spatial structure, which is consistent with the row-major limitation noted in section 8).

The second block prints where val mIoU actually peaked versus where val loss bottomed, per fold.
That is the diagnostic that tells you whether the regularization changes were load-bearing or
cosmetic — and you can run the same code over the U-Net `history_*.csv` files to settle the
question about those two screenshots.

In [ ]:
import pandas as pd
rows = []
for mode in ["block", "random"]:
    for k in range(cfg.N_FOLDS):
        repo = f"{cfg.HF_USER}/{cfg.HF_PREFIX}-{mode}-fold{k}"
        try:
            rows.append(pd.read_csv(hf_hub_download(repo, f"results_{mode}_fold{k}.csv", token=HF_TOKEN)))
        except Exception as e:
            print(f"  missing {mode} fold {k}: {type(e).__name__}")
if rows:
    df = pd.concat(rows, ignore_index=True)
    df.to_csv(f"{cfg.OUT_DIR}/aggregate_segformer_b0.csv", index=False)
    num = df.select_dtypes("number").columns
    agg = df.groupby("split_mode")[num].agg(["mean", "std"])
    print(agg[[c for c in [("mIoU","mean"),("mIoU","std"),("mF1","mean"),("kappa","mean")] if c in agg]])
    if {"block", "random"} <= set(df.split_mode):
        b = df[df.split_mode == "block"].mIoU.mean(); r = df[df.split_mode == "random"].mIoU.mean()
        print(f"\nLEAKAGE GAP (random - block) mIoU: {r-b:+.4f}  [block {b:.4f} | random {r:.4f}]")

# ---- overfitting diagnostic: where did each curve actually turn? ----
print("\nval-loss minimum vs val-mIoU maximum per fold")
for mode in ["block", "random"]:
    for k in range(cfg.N_FOLDS):
        repo = f"{cfg.HF_USER}/{cfg.HF_PREFIX}-{mode}-fold{k}"
        try:
            h = pd.read_csv(hf_hub_download(repo, f"history_{mode}_fold{k}.csv", token=HF_TOKEN))
        except Exception:
            continue
        el, em = int(h.val_loss.idxmin()), int(h.val_mIoU.idxmax())
        print(f"  {mode} f{k}: val_loss min @ e{el} ({h.val_loss.min():.3f}) | "
              f"val_mIoU max @ e{em} ({h.val_mIoU.max():.4f}) | "
              f"mIoU at loss-min e{el} = {h.val_mIoU[el]:.4f}")

## 22. Standalone inference from Hugging Face

Self-contained: point `INFER_REPO` at any fold's repo, give it images, get predictions. Nothing
above this cell is required — the model is rebuilt from the **full HF config embedded in `best.pt`**
(not re-downloaded from `nvidia/...`), so it works offline for the model definition and reconstructs
exactly the architecture that was trained, dropout rates included. Paste it into a fresh notebook
as-is.

Inputs are raw full-resolution tiles; the same `INTER_AREA` downsample and the same H/4 -> 512
logit upsample as training are applied, so inference matches training preprocessing exactly.

In [ ]:
def load_from_hf(repo_id, token=None, device="cuda"):
    import torch, numpy as np, cv2
    import torch.nn.functional as F
    from transformers import SegformerForSemanticSegmentation, SegformerConfig
    from huggingface_hub import hf_hub_download
    ck = torch.load(hf_hub_download(repo_id, "best.pt", token=token), map_location=device)
    m = SegformerForSemanticSegmentation(SegformerConfig.from_dict(ck["model_config"])).to(device).eval()
    m.load_state_dict(ck["model_state"])
    return m, ck["cfg"], ck["metrics"]

@torch.no_grad()
def predict_paths(model, c, paths, device="cuda", batch=8):
    mean = np.array([0.485, 0.456, 0.406], np.float32); std = np.array([0.229, 0.224, 0.225], np.float32)
    out = []
    for i in range(0, len(paths), batch):
        xs = []
        for p in paths[i:i + batch]:
            im = cv2.imread(str(p), cv2.IMREAD_COLOR)
            im = cv2.resize(im, (c["IMG_SIZE"], c["IMG_SIZE"]), interpolation=cv2.INTER_AREA)
            a = cv2.cvtColor(im, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
            xs.append(torch.from_numpy(((a - mean) / std).transpose(2, 0, 1)))
        x = torch.stack(xs).to(device)
        with torch.cuda.amp.autocast():
            lo = model(pixel_values=x).logits
            lo = F.interpolate(lo, size=x.shape[-2:], mode="bilinear", align_corners=False)
            out.append(lo.float().argmax(1).cpu().numpy())
    return np.concatenate(out)

# --- example ---
INFER_REPO = f"{cfg.HF_USER}/{cfg.HF_PREFIX}-block-fold0"
inf_model, inf_cfg, inf_metrics = load_from_hf(INFER_REPO, token=HF_TOKEN)
print(f"loaded {INFER_REPO} | val mIoU {inf_metrics['mIoU']:.4f}")
demo = [IMG_DIR / r[2] for r in va_recs[:4]]
preds = predict_paths(inf_model, inf_cfg, demo)
print("pred shape:", preds.shape, "| classes present:", np.unique(preds).tolist())

## 23. Qualitative triptychs — image / GT / prediction

The palette is your BGR table reversed to RGB for matplotlib; ignore renders black. The selection is
deliberately not random: it shows the highest- and lowest-mIoU validation tiles, because the failure
cases are what a reviewer wants to see. Watch specifically for the two confusions you flagged —
Unused Land vs Agricultural (spectrally similar) and urban trees vs Forest — plus Road fragmentation,
which is where clDice and IoU will disagree.

Worth comparing side by side with the U-Net figures: SegFormer's H/4 logits mean its predictions are
smoother at object boundaries, so expect the two models to fail *differently* on 10-20 px roads —
the U-Net fragmenting, B0 thickening or dropping them entirely. That contrast is a paper figure.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

PALETTE_RGB = np.zeros((256, 3), np.uint8)
for bgr, idx in BGR_TO_CLASS.items():
    PALETTE_RGB[idx] = bgr[::-1]                       # BGR table -> RGB for matplotlib
PALETTE_RGB[255] = (0, 0, 0)

def colorize(lab): return PALETTE_RGB[lab]

@torch.no_grad()
def per_image_miou(recs, k=3):
    scores = []
    for j in range(0, len(recs), 16):
        chunk = recs[j:j + 16]
        xs, ys = zip(*[val_ds[val_ds.recs.index(r)] for r in chunk])
        x = torch.stack(xs).cuda(); y = torch.stack(ys).cuda()
        with torch.cuda.amp.autocast(enabled=cfg.AMP):
            p = forward_logits(best_model, x).float().argmax(1)
        for b, r in enumerate(chunk):
            c = ConfMat(cfg.N_CLASSES); c.update(p[b], y[b])
            scores.append((metrics_from_cm(c.numpy(), cfg.CLASS_NAMES)["mIoU"], r))
    scores.sort(key=lambda t: t[0])
    return scores[-k:][::-1], scores[:k]

best_imgs, worst_imgs = per_image_miou(va_recs, k=3)

def triptych(items, title):
    fig, axes = plt.subplots(len(items), 3, figsize=(13, 4.3 * len(items)))
    axes = np.atleast_2d(axes)
    for r, (score, rec) in enumerate(items):
        _, _, iname, mname = rec
        img = cv2.cvtColor(cv2.imread(str(CACHE / "images" / iname)), cv2.COLOR_BGR2RGB)
        gt = cv2.imread(str(CACHE / "labels" / mname), cv2.IMREAD_GRAYSCALE)
        x, _ = val_ds[val_ds.recs.index(rec)]
        with torch.no_grad(), torch.cuda.amp.autocast(enabled=cfg.AMP):
            pr = forward_logits(best_model, x[None].cuda()).float().argmax(1)[0].cpu().numpy().astype(np.uint8)
        pr_vis = np.where(gt == 255, 255, pr).astype(np.uint8)   # blank ignore for fair viewing
        for c, (im, t) in enumerate([(img, iname), (colorize(gt), "ground truth"),
                                     (colorize(pr_vis), f"prediction (mIoU {score:.3f})")]):
            axes[r, c].imshow(im); axes[r, c].set_title(t, fontsize=9); axes[r, c].axis("off")
    handles = [Patch(color=PALETTE_RGB[i] / 255, label=n) for i, n in enumerate(cfg.CLASS_NAMES)]
    handles.append(Patch(color="black", label="ignore/unannotated"))
    fig.legend(handles=handles, loc="lower center", ncol=7, fontsize=8, frameon=False)
    fig.suptitle(f"{title} - SegFormer-B0, {cfg.SPLIT_MODE} fold {cfg.FOLD}", fontsize=12)
    plt.tight_layout(rect=[0, 0.04, 1, 0.97])
    p = f"{cfg.OUT_DIR}/qual_{title.replace(' ','_')}_{tag}.png"
    plt.savefig(p, dpi=130, bbox_inches="tight"); plt.show()
    hf_push(p, os.path.basename(p), "qualitative")

triptych(best_imgs, "best cases")
triptych(worst_imgs, "worst cases")

## 24. Confusion matrix figure

In [ ]:
cmn = np.array(final["confusion_matrix_norm"])
fig, ax = plt.subplots(figsize=(7.2, 6))
im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(6)); ax.set_yticks(range(6))
ax.set_xticklabels(cfg.CLASS_NAMES, rotation=45, ha="right"); ax.set_yticklabels(cfg.CLASS_NAMES)
ax.set_xlabel("Predicted"); ax.set_ylabel("Ground truth")
ax.set_title(f"Row-normalized confusion - SegFormer-B0, {cfg.SPLIT_MODE} fold {cfg.FOLD}")
for i in range(6):
    for j in range(6):
        ax.text(j, i, f"{cmn[i,j]:.2f}", ha="center", va="center", fontsize=8,
                color="white" if cmn[i, j] > 0.5 else "black")
fig.colorbar(im, fraction=0.046)
plt.tight_layout()
p = f"{cfg.OUT_DIR}/confusion_{tag}.png"
plt.savefig(p, dpi=150, bbox_inches="tight"); plt.show()
hf_push(p, os.path.basename(p), "confusion matrix")

## 25. Run checklist

1. Internet **ON**; Secrets `HF_TOKEN` (write) + `WANDB_API_KEY`; set `cfg.HF_USER`.
2. Section 5 must print **channel-order check PASSED** before you go further.
3. If the U-Net's 512 cache is already saved as a Kaggle dataset, set `PREBUILT_CACHE` to it and
   section 6 is a no-op. Otherwise let it repack once (~15-20 min), then `Save Version` and attach.
4. Section 12 must print `logits after upsample: (1, 6, 512, 512)`. If it prints 128, the upsample
   is not wired in and every metric below is computed at the wrong resolution.
5. Run the six fold-runs one per session, changing only `SPLIT_MODE` and `FOLD`:
   `block/0, block/1, block/2, random/0, random/1, random/2`. ~1.0-1.7 h each, ~6-10 h total.
6. Re-running the same fold after a timeout resumes automatically from `last.pt`.
7. After all six, run section 21 for the aggregate table, the leakage gap, and the
   loss-min-vs-mIoU-max diagnostic.

**Before you start fold 0, settle the protocol question.** `LABEL_SMOOTHING`, `COS_HORIZON_EPOCHS`
and `USE_EMA` are shared-protocol fields. Either:
- **(a)** keep them as set here and re-run every already-finished baseline (U-Net, and whatever else
  is done) with the same three values — one extra U-Net sweep is ~20 GPU-h; or
- **(b)** set `LABEL_SMOOTHING=0.0`, `COS_HORIZON_EPOCHS=200`, `USE_EMA=False` to match the existing
  U-Net exactly, and rely only on the per-architecture regularization (wd 0.01, drop_path, head
  dropout), which needs no re-runs at all.

Both are defensible; **(b) is the cheap one and (a) is the better paper.** What is not defensible is
running B0 with smoothing + EMA and comparing it against a U-Net trained without them — the table
would be measuring the regularizer, not the architecture. Decide now, write the choice into the
methods section, and keep it fixed for the remaining five baselines.

**If B0 still shows the U-Net's divergence pattern** (train loss under 0.1 while val mIoU has been
flat for 15+ epochs): raise `DROP_PATH_RATE` to 0.2 and `CLASSIFIER_DROPOUT` to 0.2 first, since
those are per-architecture and cost you nothing in comparability. Only after that consider touching
augmentation, and if you do, it changes for all seven.